## Creating Azure ML Pipelines

### Setup the Azure ML Client

In [ ]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

### Creating Folder to store scripts, data and pipeline config

In [ ]:
import os
# Create a folder for the pipeline step files
experiment_folder = 'diabetes_pipeline'
os.makedirs(experiment_folder, exist_ok=True)

print(experiment_folder)

### Creating the Data Processing Script

In [ ]:
%%writefile $experiment_folder/prep_diabetes.py

# import libraries
import os
import argparse
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import mlflow

# Get parameters
parser = argparse.ArgumentParser()
parser.add_argument("--input-data", type=str, help="input dataset")
parser.add_argument("--prepped-data", type=str, help="output folder")
args = parser.parse_args()

save_folder = args.prepped_data

with mlflow.start_run():

    print("Loading Data...")

    # Load MLTable dataset
    file_path = os.path.join(args.input_data)
    diabetes = pd.read_csv(file_path)

    # Log raw row count
    raw_rows = len(diabetes)
    mlflow.log_metric("raw_rows", raw_rows)

    # remove nulls
    diabetes = diabetes.dropna()

    # Normalize numeric columns
    scaler = MinMaxScaler()

    num_cols = [
        "Pregnancies",
        "Glucose",
        "BloodPressure",
        "SkinThickness",
        "Insulin",
        "BMI",
        "DiabetesPedigreeFunction",
        "Age"
    ]

    diabetes[num_cols] = scaler.fit_transform(diabetes[num_cols])

    # Log processed rows
    processed_rows = len(diabetes)
    mlflow.log_metric("processed_rows", processed_rows)

    # Save the prepped data
    print("Saving Data...")
    os.makedirs(save_folder, exist_ok=True)

    save_path = os.path.join(save_folder, "data.csv")
    diabetes.to_csv(save_path, index=False)

print("Data preparation complete.")

### Creating the Model Training Script

In [ ]:
%%writefile $experiment_folder/train_diabetes.py

# Import libraries
import mlflow
import argparse
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

with mlflow.start_run():
    # Get parameters
    parser = argparse.ArgumentParser()
    parser.add_argument("--training-folder", type=str, dest='training_folder', help='training data folder')
    args = parser.parse_args()
    training_folder = args.training_folder

    # load the prepared data file in the training folder
    print("Loading Data...")
    file_path = os.path.join(training_folder,'data.csv')
    diabetes = pd.read_csv(file_path)

    # Separate features and labels
    X, y = diabetes[['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age']].values, diabetes['Outcome'].values
    
    # Split data into training set and test set
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

    # Train adecision tree model
    print('Training a decision tree model...')
    model = DecisionTreeClassifier().fit(X_train, y_train)

    # calculate accuracy
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)
    print('Accuracy:', acc)
    mlflow.log_metric('Accuracy', np.float(acc))

    # calculate AUC
    y_scores = model.predict_proba(X_test)
    auc = roc_auc_score(y_test,y_scores[:,1])
    print('AUC: ' + str(auc))
    mlflow.log_metric('AUC', np.float(auc))

    # plot ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_scores[:, 1])

    fig = plt.figure(figsize=(6, 4))

    # Plot diagonal 50% baseline
    plt.plot([0, 1], [0, 1], "k--")

    # Plot model ROC
    plt.plot(fpr, tpr)

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")

    # Log figure to MLflow
    mlflow.log_figure(fig, "roc_curve.png")

    plt.show()

    # Save the trained model in the outputs folder
    print("Saving model...")
    os.makedirs('outputs', exist_ok=True)
    model_file = os.path.join('outputs', 'diabetes_model.pkl')
    joblib.dump(value=model, filename=model_file)



### Prepare a Compute Environment for the Pipeline Steps

In [ ]:
compute_target_name = "your-compute-target-name"

# Get existing compute
pipeline_cluster = ml_client.compute.get(compute_target_name)

print("Found existing compute target instance, use it.")

### Set Pipeline run configurations

In [ ]:
environment = "AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest"
compute_target = pipeline_cluster.name

print("Run configuration created.")

### Create and Run a Pipeline

In [ ]:
from azure.ai.ml import command, Input, Output
from azure.ai.ml.dsl import pipeline

# Get dataset
diabetes_ds = ml_client.data.get("diabetes-data", version="1")


# Define prep component
prep_component = command(
    name="prep_diabetes",
    code=experiment_folder,
    command="python prep_diabetes.py --input-data ${{inputs.raw_data}} --prepped-data ${{outputs.prepped_data}}",
    inputs={
        "raw_data": Input(type="uri_file")
    },
    outputs={
        "prepped_data": Output(type="uri_folder")
    },
    environment=environment,
    compute=pipeline_cluster.name
)

# Define training component
train_component = command(
    name="train_diabetes",
    code=experiment_folder,
    command="python train_diabetes.py --training-folder ${{inputs.training_data}}",
    inputs={
        "training_data": Input(type="uri_folder")
    },
    environment=environment,
    compute=pipeline_cluster.name
)


@pipeline(
    description="Diabetes training pipeline"
)
def diabetes_pipeline():

    # Step 1
    prep_job = prep_component(
        raw_data=diabetes_ds.id
    )

    # Step 2
    train_job = train_component(
        training_data=prep_job.outputs.prepped_data
    )

    return {
        "prepared_data": prep_job.outputs.prepped_data
    }


pipeline_job = diabetes_pipeline()
pipeline_job.experiment_name = "diabetes-pipeline"

ml_client.jobs.create_or_update(pipeline_job)